# Data Understanding & Quality Audit

In this notebook, we connect to the raw SQLite tables to audit the synthetic dataset for data quality issues before analysis.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../db/churn.db')

## 1. Finding Duplicates in Customers

In [2]:
df_cust = pd.read_sql("SELECT * FROM raw_customers", conn)
dupes = df_cust[df_cust.duplicated('customer_id', keep=False)].sort_values('customer_id')
print(f"Found {len(dupes)} duplicate rows representing {dupes['customer_id'].nunique()} customers.")
dupes.head()

                                customer_id  ... company_size_band
13075  028bce35-39a9-44a1-b449-90638a08bbc7  ...              Solo
30118  028bce35-39a9-44a1-b449-90638a08bbc7  ...              Solo
30081  02ce56f1-3dfb-4f43-a933-23f41d767398  ...              2-10
17963  02ce56f1-3dfb-4f43-a933-23f41d767398  ...              2-10
30033  03619c84-4109-401b-b0b3-283a9d76490e  ...              2-10

[5 rows x 6 columns]

Found 300 duplicate rows representing 150 customers.


## 2. Inconsistent Categories (City)

In [3]:
city_counts = df_cust['city'].value_counts()
print("City value counts showing inconsistent casing/spelling:")
print(city_counts.tail(15))

City value counts showing inconsistent casing/spelling:
city
KOLKATA      313
pune         309
lucknow      300
kolkata      299
mumbai       299
JAIPUR       296
MUMBAI       294
bengaluru    294
ahmedabad    286
PUNE         286
hyderabad    285
delhi        283
chennai      278
CHENNAI      275
Bombay       156
Name: count, dtype: int64


## 3. Missing Acquisition Channels

In [4]:
missing_channel = df_cust['acquisition_channel'].isna().sum()
print(f"Customers missing acquisition channel: {missing_channel} ({missing_channel/len(df_cust)*100:.2f}%)")

Customers missing acquisition channel: 619 (2.05%)


## 4. Logical Errors in Subscriptions

In [5]:
df_subs = pd.read_sql("SELECT * FROM raw_subscriptions", conn)
logic_errs = df_subs[df_subs['end_date'] < df_subs['start_date']]
print(f"Subscriptions with end_date before start_date: {len(logic_errs)}")
logic_errs.head()

                            subscription_id  ...           cancel_reason
11181  82f3e810-2a63-411f-a8ef-7ab46f8be4c6  ...  Switched to competitor
18885  51b6e8c2-c6b7-4cbc-bb55-638df275966e  ...  Switched to competitor
22886  daff9d16-cf31-470f-b216-895778d09181  ...        Missing features
28683  1f096f50-4d7e-4955-9401-78a2a7ecf596  ...           Too expensive
29368  b0848afb-147c-4ab7-ac7f-4110afd760bb  ...        Missing features

[5 rows x 9 columns]

Subscriptions with end_date before start_date: 5


## 5. Orphan Payments

In [6]:
df_payments = pd.read_sql("SELECT * FROM raw_payments", conn)
orphans = df_payments[~df_payments['customer_id'].isin(df_cust['customer_id'])]
print(f"Orphan payment records: {len(orphans)}")
orphans.head()

                                  payment_id  ...  status
188565  a5f23457-628c-4b69-836c-43966c0f267a  ...  Failed
188566  9f513eec-f471-4fcb-9e92-1ba47c06725c  ...  Failed
188567  8f022628-2e81-42b9-a29b-79f39877e19c  ...  Failed
188568  da506f9f-e0d8-4a01-b4d0-af6fa17b473e  ...  Failed
188569  36631e23-d1df-4109-9af2-30b83afc10f7  ...  Failed

[5 rows x 5 columns]

Orphan payment records: 100


## Summary of Fixes Required
1. **Customers**: Deduplicate by `customer_id`, picking the first row. Standardize city casing/names (e.g. bombay -> Mumbai). Impute missing channels with 'Unknown'.
2. **Subscriptions**: Nullify or drop the `end_date` where `end_date < start_date`.
3. **Payments**: Drop records where `customer_id` doesn't exist in the customers table.
4. **Usage**: Cap negative `session_minutes` to 0.

In [7]:
conn.close()